# 01 — Dữ liệu và PIT

Chứng minh đúng input, config, code và các mẫu financial state theo ngày quyết định. Không fit mô hình, không thu thập lại dữ liệu.

**Đầu vào:** `M3/sources.json`, hai run/seal Dev và Holdout, bảy input đã pin. **Đầu ra:** các bảng dưới đây và `01_verification.json` trong thư mục reproduction.

Hash sai, source thay đổi hoặc target mẫu không khớp sẽ dừng bằng assertion. `m3c.py` chỉ điều phối; quy tắc PIT/strategy nằm trong module chính thức.

In [1]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "M3/sources.json").is_file())
sys.path.insert(0, str(root / "M3"))
from m3c import Context, table, NAMES
ctx = Context()
print("Code nguồn:", ctx.identity["source_commit"])
print("Code thực thi:", ctx.identity["execution_commit"])
print("Config và nguồn canonical: PASS")
print("Evidence:", ctx.output)


Code nguồn: c9c1f5f0c88f2dbb3ebbb50effa47003b24b66cf
Code thực thi: c9c1f5f0c88f2dbb3ebbb50effa47003b24b66cf
Config và nguồn canonical: PASS
Evidence: C:\Users\HP\Downloads\Phân cụm động lượng TTCK\M3\reproduction\20261104T145912988868Z


## 1. Identity và hash đầu vào

Runner hash đầy đủ bảy input một lần trong phiên, đồng thời lưu snapshot trần/sàn đã kiểm chứng. Mỗi kernel kiểm tra fingerprint để phát hiện input thay đổi; mở notebook riêng không có evidence sẽ chạy preflight mới. Hai seal và đủ 33 file mỗi run được kiểm tra lại. B-PIT metadata hiện hành cũng được ghi hash; không build B-PIT lại.

In [2]:
table(ctx.provenance(), ["input", "path", "sha256", "size"])
table([dict(segment=s, run_id=v["manifest"]["run_id"], **v["manifest"]["counts"]["combined"]) for s,v in ctx.runs.items()])

input,path,sha256,size
terminal_sha256,data/web/delta-intelligence-v4/terminal.sqlite,0662585b95a7d529854dbcea8816e47e24466dc55ff1368cbe923c018c5670e9,526651392
indicators_sha256,data/financial/indicators_v1/results/indicators-2c4a24e3830d325f/indicators.sqlite,26bfc2d922a8f39f1fb8234e6b50c788ef83ac4b229059ed339a14ebd1213023,111276032
availability_sha256,data/financial/disclosure_dates_v1/availability-4f0f83108e761dae/availability.csv,4f0f83108e761daeb86289e290de3aff0379abbf1ed630ac31c842146186648f,873745
c8_benchmark_sha256,artifacts/cafef_primary/cafef-c8-complete-only-v2/canonical/benchmark_daily.jsonl,75fc214cb079159fe2df03f29fe9c8075705b24cbb114f09f0e800bf3c4285a2,954587
c8_calendar_sha256,artifacts/cafef_primary/cafef-c8-complete-only-v2/canonical/trading_calendar.jsonl,9aa4ffd869052654dc5b9ed7f6b19c8339469057fa80dda2a66f9a6aecb68b6f,2864640
c8_prices_sha256,artifacts/cafef_primary/cafef-c8-complete-only-v2/canonical/prices_daily.jsonl,20ecee90df48869fa4ed7196c64ff857775ebd54fcaeb1b93ced76d6635a2b1f,4193211376
b_pit_labels_sha256,data/financial/b_pit_v1/bpit-4a9d2e87f0de84ec/labels.csv,4a9d2e87f0de84ec9e80364b24043cb6091aa4f34f7dcfd187d8526e9f44579a,225357


segment,run_id,decisions,executed,too_small,not_executed_segment_end
development,20261104T121107Z-5557288c,60,59,13,1
holdout,20261104T133507Z-5557288c,24,23,1,1


## 2. Tái lập PIT trên một số decision dates

Dùng `Strategies.target_for` và `universe.financial_state` chính thức, kiểm tra FY mới nhất usable, stale và model cutoff. Bảng minh họa lấy tối đa ba ứng viên mỗi ngày; assertion kiểm tra toàn bộ ứng viên trong các ngày mẫu. Target mẫu đối chiếu cả mã, rank, trọng số và status với CSV canonical.

FY trước 2022 dùng nguồn LAG_RULE, từ FY2022 dùng CAFEF_DISCLOSURE theo policy đã khóa. Ngày usable là phiên kế tiếp sau ngày công bố, không suy giờ.

In [3]:
samples, checks = ctx.pit_samples()
table(checks)
table(samples)

segment,decision_date,candidates,checked_eligible_rows,target
development,2019-01-31,63,63,PASS
development,2020-04-29,6,6,PASS
development,2021-04-29,30,30,PASS
development,2021-07-30,11,11,PASS
development,2022-04-29,3,3,PASS
development,2023-04-28,12,12,PASS
development,2023-12-29,80,80,PASS
holdout,2024-01-31,97,97,PASS
holdout,2025-01-24,12,12,PASS
holdout,2025-04-29,4,4,PASS


segment,decision_date,ticker,financial_year,usable_from_date,cutoff,result
development,2019-01-31,AGM,2017,2018-07-02,2018-07-02,PASS
development,2019-01-31,AMV,2017,2018-07-02,2018-07-02,PASS
development,2019-01-31,CAP,2017,2018-07-02,2018-07-02,PASS
development,2020-04-29,D11,2018,2019-07-01,2019-07-01,PASS
development,2020-04-29,GVT,2018,2019-07-01,2019-07-01,PASS
development,2020-04-29,IDV,2018,2019-07-01,2019-07-01,PASS
development,2021-04-29,BVG,2019,2020-07-01,2020-07-01,PASS
development,2021-04-29,CMT,2019,2020-07-01,2020-07-01,PASS
development,2021-04-29,DGW,2019,2020-07-01,2020-07-01,PASS
development,2021-07-30,CBI,2020,2021-07-01,2021-07-01,PASS


## 3. Giới hạn của kết luận PIT

PASS chứng minh việc dùng availability/cutoff đúng theo snapshot đã pin. Không xác nhận dữ liệu tài chính là vintage nguyên thủy hoặc không có restatement. Các giới hạn VCG/SBT được owner chấp nhận vẫn áp dụng; không mở lại audit PDF trong M3-C.

In [4]:
print(ctx.finish())
print("NOTEBOOK_01_PASS")

PASS — nguồn canonical giữ nguyên; không có Final performance.
NOTEBOOK_01_PASS
